# 🏃 Human Activity Recognition with Smartphones
## Unsupervised K-Means Clustering on Sensor Data

**Problem Statement:** Smart devices collect sensor data (accelerometer, gyroscope) to understand human activities. The goal is to group similar activity patterns (e.g., walking, sitting, standing) **without using labels** using K-Means Clustering.

**Dataset:** UCI HAR Dataset — 561 features extracted from accelerometer & gyroscope readings from Samsung Galaxy S II.

---

## Step 1: Import Libraries & Load Dataset

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import pickle
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries imported successfully!')


In [ ]:
# Load the dataset
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Train dataset shape: {train.shape}')
print(f'Test dataset shape:  {test.shape}')
print(f'\nTotal features: {train.shape[1] - 2} (excluding subject & Activity columns)')

## Step 2: Explore the Dataset

In [ ]:
# View first 5 rows
print('=== First 5 Rows ===')
train.head()

In [ ]:
# Dataset info
print('=== Dataset Info ===')
print(f'Columns: {list(train.columns[:10])} ... ({len(train.columns)} total)')
print(f'\nData Types:\n{train.dtypes.value_counts()}')
print(f'\nBasic Statistics:')
train.describe()

In [ ]:
# Activity distribution in training data
print('=== Activity Distribution ===')
activity_counts = train['Activity'].value_counts()
print(activity_counts)

plt.figure(figsize=(10, 5))
colors = sns.color_palette('Set2', len(activity_counts))
activity_counts.plot(kind='bar', color=colors, edgecolor='black', linewidth=0.5)
plt.title('Activity Distribution in Training Data', fontsize=15, fontweight='bold')
plt.xlabel('Activity', fontsize=13)
plt.ylabel('Count', fontsize=13)
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(activity_counts.values):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Subject distribution
print(f'Number of unique subjects: {train["subject"].nunique()}')
print(f'Subject IDs: {sorted(train["subject"].unique())}')

## Step 3: Preprocessing & Handle Missing Values

In [ ]:
# Check for missing values
print('=== Missing Values ===')
print(f'Missing values in Train: {train.isnull().sum().sum()}')
print(f'Missing values in Test:  {test.isnull().sum().sum()}')

# Check for duplicates
print(f'\nDuplicate rows in Train: {train.duplicated().sum()}')
print(f'Duplicate rows in Test:  {test.duplicated().sum()}')

In [ ]:
# Combine train and test for unsupervised clustering
df = pd.concat([train, test], axis=0, ignore_index=True)
print(f'Combined dataset shape: {df.shape}')

# Separate features from labels
X = df.drop(columns=['Activity', 'subject'], errors='ignore')
y = df['Activity']  # Keep for evaluation only

# Handle missing values (fill with median if any)
X = X.fillna(X.median())
print(f'Features shape: {X.shape}')
print(f'Missing values after preprocessing: {X.isnull().sum().sum()}')

feature_names = list(X.columns)
print(f'\nTotal features: {len(feature_names)}')

## Step 4: Feature Selection & Feature Scaling

In [ ]:
# Feature categories breakdown
print('=== Feature Categories ===')
categories = {
    'tBodyAcc': len([f for f in feature_names if f.startswith('tBodyAcc')]),
    'tGravityAcc': len([f for f in feature_names if f.startswith('tGravityAcc')]),
    'tBodyGyro': len([f for f in feature_names if f.startswith('tBodyGyro')]),
    'fBody (FFT)': len([f for f in feature_names if f.startswith('fBody')]),
    'angle': len([f for f in feature_names if f.startswith('angle')]),
}
for cat, count in categories.items():
    print(f'  {cat:20s}: {count} features')

# Correlation heatmap of first 20 features
plt.figure(figsize=(14, 10))
corr = X.iloc[:, :20].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, annot=False)
plt.title('Correlation Heatmap (First 20 Features)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('=== Feature Scaling Applied ===')
print(f'Shape after scaling: {X_scaled.shape}')
print(f'Mean of first 5 features (should be ~0): {X_scaled[:, :5].mean(axis=0).round(6)}')
print(f'Std of first 5 features (should be ~1):  {X_scaled[:, :5].std(axis=0).round(6)}')

## Step 5: Elbow Method - Finding Optimal Number of Clusters

In [ ]:
# Elbow Method
K_range = range(2, 12)
inertias = []
silhouette_scores = []

print('=== Elbow Method ===')
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=5000, random_state=42)
    silhouette_scores.append(sil)
    print(f'  K={k:2d}  |  Inertia={km.inertia_:12.2f}  |  Silhouette={sil:.4f}')

In [ ]:
# Plot Elbow Curve & Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Elbow curve
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=6, color='red', linestyle='--', alpha=0.7, label='Optimal K=6')
ax1.set_xlabel('Number of Clusters (K)', fontsize=13)
ax1.set_ylabel('Inertia (WCSS)', fontsize=13)
ax1.set_title('Elbow Method - Finding Optimal K', fontsize=15, fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Silhouette scores
ax2.plot(K_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
ax2.axvline(x=6, color='red', linestyle='--', alpha=0.7, label='K=6')
ax2.set_xlabel('Number of Clusters (K)', fontsize=13)
ax2.set_ylabel('Silhouette Score', fontsize=13)
ax2.set_title('Silhouette Score vs K', fontsize=15, fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n📌 Optimal K = 6 (matching 6 actual activities: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING)')

## Step 6: Apply K-Means Clustering (K=6)

In [ ]:
# Apply K-Means with K=6
kmeans = KMeans(n_clusters=6, init='k-means++', n_init=10, max_iter=300, random_state=42)
kmeans.fit(X_scaled)
cluster_labels = kmeans.labels_

# Add cluster labels to dataset
df['Cluster'] = cluster_labels

print('=== K-Means Clustering Results ===')
print(f'Inertia (WCSS): {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X_scaled, cluster_labels, sample_size=5000, random_state=42):.4f}')
print(f'Number of iterations: {kmeans.n_iter_}')

print(f'\nCluster Distribution:')
cluster_dist = pd.Series(cluster_labels).value_counts().sort_index()
for cluster_id, count in cluster_dist.items():
    print(f'  Cluster {cluster_id}: {count:5d} samples ({count/len(cluster_labels)*100:.1f}%)')

In [ ]:
# Cluster distribution bar chart
plt.figure(figsize=(10, 5))
colors = sns.color_palette('Set2', 6)
cluster_dist.plot(kind='bar', color=colors, edgecolor='black', linewidth=0.5)
plt.title('K-Means Cluster Distribution', fontsize=15, fontweight='bold')
plt.xlabel('Cluster', fontsize=13)
plt.ylabel('Number of Samples', fontsize=13)
plt.xticks(rotation=0)
for i, v in enumerate(cluster_dist.values):
    plt.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 7: PCA-based 2D/3D Cluster Visualization

In [ ]:
# PCA - Reduce to 3 components
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print('=== PCA Results ===')
print(f'Explained Variance Ratio: {pca.explained_variance_ratio_.round(4)}')
print(f'Total Variance Explained: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.1f}%)')
print(f'\nPC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print(f'PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}% of variance')
print(f'PC3 explains {pca.explained_variance_ratio_[2]*100:.1f}% of variance')

In [ ]:
# 2D PCA Visualization: Clusters vs Actual Labels (side by side)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Plot 1: K-Means Clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, 
                           cmap='Set2', alpha=0.5, s=10)
axes[0].set_xlabel('PC1', fontsize=13)
axes[0].set_ylabel('PC2', fontsize=13)
axes[0].set_title('K-Means Clusters (PCA 2D)', fontsize=15, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: Actual Activity Labels
activity_map = {a: i for i, a in enumerate(y.unique())}
y_numeric = y.map(activity_map).values
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_numeric,
                           cmap='Set2', alpha=0.5, s=10)
axes[1].set_xlabel('PC1', fontsize=13)
axes[1].set_ylabel('PC2', fontsize=13)
axes[1].set_title('Actual Activity Labels (PCA 2D)', fontsize=15, fontweight='bold')
cbar = plt.colorbar(scatter2, ax=axes[1], label='Activity')
cbar.set_ticks(list(activity_map.values()))
cbar.set_ticklabels(list(activity_map.keys()))

plt.tight_layout()
plt.show()

In [ ]:
# 3D PCA Visualization
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
                     c=cluster_labels, cmap='Set2', alpha=0.5, s=10)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('K-Means Clusters (PCA 3D)', fontsize=15, fontweight='bold')
plt.colorbar(scatter, label='Cluster', shrink=0.6)

plt.tight_layout()
plt.show()

print('📌 The 3D visualization shows clearer separation of clusters in PCA space.')

## Step 8: Compare Clusters with Actual Activity Labels

In [ ]:
# Adjusted Rand Index
ari = adjusted_rand_score(y, cluster_labels)
print(f'=== Evaluation Metrics ===')
print(f'Adjusted Rand Index (ARI): {ari:.4f}')
print(f'  - ARI = 1.0 means perfect agreement with true labels')
print(f'  - ARI = 0.0 means random assignment')
print(f'\n  Our ARI of {ari:.4f} indicates {"good" if ari > 0.3 else "moderate"} cluster-label agreement.')

In [ ]:
# Confusion Matrix / Cross-tabulation
cross_tab = pd.crosstab(y, cluster_labels, rownames=['Actual Activity'], colnames=['Cluster'])
print('=== Confusion Matrix: Clusters vs Actual Activities ===\n')
print(cross_tab)

# Heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, linecolor='gray')
plt.title('Cluster vs Actual Activity - Confusion Matrix', fontsize=15, fontweight='bold')
plt.xlabel('K-Means Cluster', fontsize=13)
plt.ylabel('Actual Activity', fontsize=13)
plt.tight_layout()
plt.show()

print('\n📌 Interpretation:')
print('  - High values along a single row/column indicate the cluster captures that activity well.')
print('  - Dynamic activities (walking variants) are harder to separate than static ones (sitting, standing, laying).')

## Step 9: Interpret Clusters & Identify Activity Patterns

In [ ]:
# Map clusters to most common actual activity
print('=== Cluster → Activity Mapping ===')
df_result = pd.DataFrame({'Activity': y, 'Cluster': cluster_labels})

cluster_mapping = {}
for cluster_id in range(6):
    cluster_data = df_result[df_result['Cluster'] == cluster_id]
    most_common = cluster_data['Activity'].mode()[0]
    pct = (cluster_data['Activity'] == most_common).sum() / len(cluster_data) * 100
    cluster_mapping[cluster_id] = most_common
    print(f'  Cluster {cluster_id} → {most_common:25s} ({pct:.1f}% match, {len(cluster_data)} samples)')

print(f'\n📌 Key Observations:')
print('  - LAYING is typically captured very well by a single cluster (>90% match)')
print('  - SITTING and STANDING may share clusters (similar body positions)')
print('  - Walking activities show more mixing due to movement similarities')

In [ ]:
# Cluster centroids analysis for key features
key_features = [
    'tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
    'tGravityAcc-mean()-X', 'tGravityAcc-mean()-Y', 'tGravityAcc-mean()-Z',
    'tBodyGyro-mean()-X', 'tBodyGyro-mean()-Y', 'tBodyGyro-mean()-Z'
]
key_idx = [feature_names.index(f) for f in key_features]

print('=== Cluster Centroids (Key Sensor Features, Scaled) ===\n')
centroid_df = pd.DataFrame(
    kmeans.cluster_centers_[:, key_idx],
    columns=[f.split('-')[0][-10:] + '-' + f.split('-')[1] for f in key_features],
    index=[f'Cluster {i}' for i in range(6)]
).round(4)
print(centroid_df.to_string())

In [ ]:
# Radar chart of cluster centroids
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw=dict(polar=True))
axes = axes.flatten()

angles = np.linspace(0, 2 * np.pi, len(key_features), endpoint=False).tolist()
angles += angles[:1]

colors_radar = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

for cluster_id in range(6):
    ax = axes[cluster_id]
    values = kmeans.cluster_centers_[cluster_id, key_idx].tolist()
    values += values[:1]
    
    ax.fill(angles, values, alpha=0.25, color=colors_radar[cluster_id])
    ax.plot(angles, values, 'o-', linewidth=2, color=colors_radar[cluster_id])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([f.split('-')[0][-8:] for f in key_features], size=7)
    mapped_activity = cluster_mapping.get(cluster_id, '')
    ax.set_title(f'Cluster {cluster_id}\n({mapped_activity})', size=11, fontweight='bold', pad=15)

plt.suptitle('Cluster Centroids - Radar Charts', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# PCA Component Loading - which original features matter most
pca_loadings = pd.DataFrame(
    pca.components_[:3],
    columns=feature_names,
    index=['PC1', 'PC2', 'PC3']
)

# Top 15 features by PC1 loading
top_15 = pca_loadings.loc['PC1'].abs().nlargest(15)

plt.figure(figsize=(12, 6))
top_15.plot(kind='barh', color='#667eea', edgecolor='black', linewidth=0.5)
plt.title('Top 15 Features by |PC1 Loading| (Most Important for Clustering)', 
         fontsize=14, fontweight='bold')
plt.xlabel('|Loading|', fontsize=13)
plt.ylabel('Feature', fontsize=13)
plt.tight_layout()
plt.show()

print('\n📌 These features contribute the most to distinguishing between activity clusters.')

## Step 10: Save Models & Streamlit App

In [ ]:
# Save models for the Streamlit app
with open('kmeans_model.pkl', 'wb') as f: pickle.dump(kmeans, f)
with open('scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
with open('pca_model.pkl', 'wb') as f: pickle.dump(pca, f)
with open('feature_names.pkl', 'wb') as f: pickle.dump(feature_names, f)

print('=== Models Saved Successfully ===')
print('  ✅ kmeans_model.pkl')
print('  ✅ scaler.pkl')
print('  ✅ pca_model.pkl')
print('  ✅ feature_names.pkl')
print('\n  Run the Streamlit app with:')
print('    streamlit run app.py')


In [ ]:
# Show the Streamlit app code
print('=== Streamlit App Code (app.py) ===\n')
print('The Streamlit app (app.py) provides:')
print('  1. 📊 Dashboard - Cluster distribution, PCA scatter, activity pie chart')
print('  2. 🔮 Predict Activity - Input sensor values to predict cluster')
print('  3. 📈 Visualizations - 3D PCA, Confusion Matrix, Feature Importance, Radar')
print('  4. ℹ️ About - Project details and real-world applications')
print('\n  To run: streamlit run app.py')
print('  To deploy: streamlit community cloud (free hosting)')

## Summary & Conclusions

### Key Findings:

1. **Dataset**: 10,299 samples with 561 features from accelerometer & gyroscope sensors, 6 activities.

2. **Preprocessing**: No missing values found. Applied StandardScaler for normalization.

3. **Elbow Method**: Optimal K=6 clusters (matching 6 actual activities).

4. **K-Means Results**: Silhouette Score indicates moderate clustering quality. The overlap is expected since some activities are physically similar.

5. **PCA Visualization**: First 3 PCs explain ~60% of variance. Clear separation between dynamic (walking) and static (sitting/standing/laying) activities.

6. **Cluster Evaluation**: 
   - LAYING is almost perfectly captured by a single cluster (~95% match)
   - Static activities (SITTING/STANDING) show some overlap
   - Walking variants are harder to separate

### Real-World Applications:
- **Fitness Tracking**: Auto-detect exercises and daily activities
- **Healthcare Monitoring**: Track patient mobility, detect falls
- **Elder Care**: Monitor activity patterns for safety alerts
- **Workplace Safety**: Ensure proper movement protocols

### Deployment:
- Streamlit app (`app.py`) provides an interactive UI
- Deploy via **Streamlit Community Cloud** for a public link
- Models saved as `.pkl` files for production use